In [ ]:
# ==============================================================================
# PARALLEL DIM: STORE
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, safe_count, generate_batch_id
import pandas as pd
from helpers.silver_transforms import transform_store_full_pipeline

logger = setup_logger("parallel_dim_store")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

dependencies = ["staff", "address", "city", "country"]

bronze_batch_id = get_latest_batch_id(spark, "store")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for store; run bronze load first.")
logger.info(f"Using bronze batch_id for store: {bronze_batch_id}")

config = TableConfig(
    table_name="store",
    business_key="store_id",
    surrogate_key="store_key",
    watermark_column="last_update",
    scd_type=2,
    tracking_columns=["store_manager_id", "store_manager_first_name", "store_manager_last_name"],
    gold_table_name="dim_store",
    silver_transform=transform_store_full_pipeline,
    dependencies=dependencies,
)

print("Row Counts (Before):")
print(f"dim_store: {safe_count(spark, 'dim_store')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_store: {safe_count(spark, 'dim_store')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
